In [1]:
import numpy as np
import pandas as pd
from f1winnerprediction import (
   config, 
	io_fastf1
)
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report,roc_auc_score, roc_curve, auc, mean_absolute_error, r2_score, accuracy_score, f1_score
import fastf1
import fastf1.core
import xgboost as xgb
from pprint import pprint

pd.set_option('display.max_columns', None)

fastf1.Cache.enable_cache(config.FASTF1_RAW_CACHE_DIR)

# Load sessions

In [2]:
sessions: dict[int, list[fastf1.core.Session]] = io_fastf1.load_sessions_from_years()
sessions

{2021: [2021 Season Round 1: Bahrain Grand Prix - Race,
  2021 Season Round 2: Emilia Romagna Grand Prix - Race,
  2021 Season Round 3: Portuguese Grand Prix - Race,
  2021 Season Round 4: Spanish Grand Prix - Race,
  2021 Season Round 5: Monaco Grand Prix - Race,
  2021 Season Round 6: Azerbaijan Grand Prix - Race,
  2021 Season Round 7: French Grand Prix - Race,
  2021 Season Round 8: Styrian Grand Prix - Race,
  2021 Season Round 9: Austrian Grand Prix - Race,
  2021 Season Round 10: British Grand Prix - Race,
  2021 Season Round 11: Hungarian Grand Prix - Race,
  2021 Season Round 12: Belgian Grand Prix - Race,
  2021 Season Round 13: Dutch Grand Prix - Race,
  2021 Season Round 14: Italian Grand Prix - Race,
  2021 Season Round 15: Russian Grand Prix - Race,
  2021 Season Round 16: Turkish Grand Prix - Race,
  2021 Season Round 17: United States Grand Prix - Race,
  2021 Season Round 18: Mexico City Grand Prix - Race,
  2021 Season Round 19: São Paulo Grand Prix - Race,
  2021 Sea

In [3]:
driver_mapping = io_fastf1.build_drivers_dict(sessions)
driver_mapping

{'OCO': {'index': 19},
 'SAI': {'index': 19},
 'VET': {'index': 21},
 'MAZ': {'index': 21},
 'MSC': {'index': 21},
 'GAS': {'index': 19},
 'HAM': {'index': 19},
 'LAT': {'index': 21},
 'ALO': {'index': 19},
 'BOT': {'index': 23},
 'NOR': {'index': 19},
 'RAI': {'index': 21},
 'GIO': {'index': 21},
 'TSU': {'index': 19},
 'RIC': {'index': 17},
 'RUS': {'index': 19},
 'STR': {'index': 19},
 'LEC': {'index': 19},
 'VER': {'index': 19},
 'PER': {'index': 23},
 'KUB': {'index': 13},
 'MAG': {'index': 23},
 'ZHO': {'index': 23},
 'ALB': {'index': 19},
 'HUL': {'index': 19},
 'DEV': {'index': 9},
 'PIA': {'index': 19},
 'SAR': {'index': 14},
 'LAW': {'index': 19},
 'BEA': {'index': 19},
 'COL': {'index': 19},
 'DOO': {'index': 5},
 'BOR': {'index': 19},
 'ANT': {'index': 19},
 'HAD': {'index': 19}}

# Prepare laptime data

In [4]:
session_last_year = sessions[2023][0]  # First session of 2023
pprint(session_last_year.laps.columns)

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')


In [5]:
mean_lap_time_last_year = session_last_year.laps["LapTime"].dt.total_seconds().groupby(session_last_year.laps["Driver"]).mean().sort_values()
df_lap_time_last_year = mean_lap_time_last_year.reset_index()
df_lap_time_last_year

,Driver,LapTime
0,VER,98.890105
1,PER,99.100404
2,ALO,99.567947
3,LEC,99.644051
4,SAI,99.733123
5,HAM,99.784439
6,STR,99.846281
7,RUS,99.870333
8,BOT,100.164614
9,GAS,100.184018


In [6]:
session_current_year = sessions[2024][0]  # First session of 2024

In [7]:
mean_lap_time_current_year = session_current_year.laps["LapTime"].dt.total_seconds().groupby(session_current_year.laps["Driver"]).mean().sort_values()
df_lap_time_current_year = mean_lap_time_current_year.reset_index()
df_lap_time_current_year

,Driver,LapTime
0,VER,96.574421
1,PER,96.968404
2,SAI,97.014947
3,LEC,97.270368
4,RUS,97.395263
5,NOR,97.424561
6,HAM,97.457298
7,PIA,97.558316
8,ALO,97.888228
9,STR,98.209789


In [8]:
data = df_lap_time_last_year.merge(df_lap_time_current_year, on="Driver", suffixes=('_2023', '_2024'))
data


,Driver,LapTime_2023,LapTime_2024
0,VER,98.890105,96.574421
1,PER,99.100404,96.968404
2,ALO,99.567947,97.888228
3,LEC,99.644051,97.270368
4,SAI,99.733123,97.014947
5,HAM,99.784439,97.457298
6,STR,99.846281,98.209789
7,RUS,99.870333,97.395263
8,BOT,100.164614,98.503745
9,GAS,100.184018,98.877839


In [12]:
def get_laps(session: fastf1.core.Session) -> pd.DataFrame:
	"""
	Extracts lap time data for all drivers from a given session.

	Args:
		session: A fastf1 session object.

	Returns:
		A pandas DataFrame with Driver, LapNumber, and LapTime in seconds.
	"""
	if not session.laps.empty:
		laps = session.laps.copy()
		laps["LapTime"] = laps["LapTime"].dt.total_seconds()
		return laps[["Driver", "LapNumber", "LapTime"]].dropna()
	return pd.DataFrame()

def get_comparison_data(current_session: fastf1.core.Session, last_year_session: fastf1.core.Session) -> pd.DataFrame:
	"""
	Merges lap times from the same event in two consecutive years.

	Args:
		current_session: The session object for the current year.
		last_year_session: The session object for the previous year.

	Returns:
		A merged DataFrame with lap times from both years aligned by driver and lap number.
	"""
	laps_current = get_laps(current_session)
	laps_last_year = get_laps(last_year_session)

	if laps_current.empty or laps_last_year.empty:
		return pd.DataFrame()

	# Merge data on Driver and LapNumber
	merged_laps = pd.merge(
		laps_last_year, 
		laps_current, 
		on=["Driver", "LapNumber"], 
		suffixes=('_last_year', '_current_year')
	)
	return merged_laps

# --- Main processing loop ---
all_laps_data = []
# Assuming config.YEARS_TO_FETCH is sorted, start from the second year
for year in tqdm(sorted(config.YEARS_TO_FETCH)[1:-1], desc="Processing Years"):
	last_year = year - 1
	
	# Create a quick lookup map for last year's sessions by event name
	last_year_session_map = {
		s.event.EventName: s for s in sessions.get(last_year, [])
	}

	for current_session in sessions.get(year, []):
		# Find the corresponding session from the previous year
		last_year_session = last_year_session_map.get(current_session.event.EventName)
		print(f"Current Year: {year}, Event: {current_session.event.EventName}")
		if last_year_session:
			comparison_df = get_comparison_data(current_session, last_year_session)
			if not comparison_df.empty:
				all_laps_data.append(comparison_df)

# Concatenate all data into a single DataFrame
if all_laps_data:
	training_data = pd.concat(all_laps_data, ignore_index=True)
	print(f"Successfully created training data with {len(training_data)} samples.")
	print(training_data.head())
else:
	print("No matching sessions found to create training data.")
	training_data = pd.DataFrame()

Processing Years:   0%|          | 0/3 [00:00<?, ?it/s]

Current Year: 2022, Event: Bahrain Grand Prix
Current Year: 2022, Event: Saudi Arabian Grand Prix
Current Year: 2022, Event: Australian Grand Prix
Current Year: 2022, Event: Emilia Romagna Grand Prix
Current Year: 2022, Event: Miami Grand Prix
Current Year: 2022, Event: Spanish Grand Prix
Current Year: 2022, Event: Monaco Grand Prix
Current Year: 2022, Event: Azerbaijan Grand Prix
Current Year: 2022, Event: Canadian Grand Prix
Current Year: 2022, Event: British Grand Prix
Current Year: 2022, Event: Austrian Grand Prix
Current Year: 2022, Event: French Grand Prix
Current Year: 2022, Event: Hungarian Grand Prix
Current Year: 2022, Event: Belgian Grand Prix
Current Year: 2022, Event: Dutch Grand Prix
Current Year: 2022, Event: Italian Grand Prix
Current Year: 2022, Event: Singapore Grand Prix
Current Year: 2022, Event: Japanese Grand Prix
Current Year: 2022, Event: United States Grand Prix
Current Year: 2022, Event: Mexico City Grand Prix
Current Year: 2022, Event: São Paulo Grand Prix
Cu

Processing Years:  33%|███▎      | 1/3 [00:00<00:00,  4.87it/s]

Current Year: 2023, Event: Bahrain Grand Prix
Current Year: 2023, Event: Saudi Arabian Grand Prix
Current Year: 2023, Event: Australian Grand Prix
Current Year: 2023, Event: Azerbaijan Grand Prix
Current Year: 2023, Event: Miami Grand Prix
Current Year: 2023, Event: Monaco Grand Prix
Current Year: 2023, Event: Spanish Grand Prix
Current Year: 2023, Event: Canadian Grand Prix
Current Year: 2023, Event: Austrian Grand Prix
Current Year: 2023, Event: British Grand Prix
Current Year: 2023, Event: Hungarian Grand Prix
Current Year: 2023, Event: Belgian Grand Prix
Current Year: 2023, Event: Dutch Grand Prix
Current Year: 2023, Event: Italian Grand Prix
Current Year: 2023, Event: Singapore Grand Prix
Current Year: 2023, Event: Japanese Grand Prix
Current Year: 2023, Event: Qatar Grand Prix
Current Year: 2023, Event: United States Grand Prix


Processing Years:  67%|██████▋   | 2/3 [00:00<00:00,  4.27it/s]

Current Year: 2023, Event: Mexico City Grand Prix
Current Year: 2023, Event: São Paulo Grand Prix
Current Year: 2023, Event: Las Vegas Grand Prix
Current Year: 2023, Event: Abu Dhabi Grand Prix
Current Year: 2024, Event: Bahrain Grand Prix
Current Year: 2024, Event: Saudi Arabian Grand Prix
Current Year: 2024, Event: Australian Grand Prix
Current Year: 2024, Event: Japanese Grand Prix
Current Year: 2024, Event: Chinese Grand Prix
Current Year: 2024, Event: Miami Grand Prix
Current Year: 2024, Event: Emilia Romagna Grand Prix
Current Year: 2024, Event: Monaco Grand Prix
Current Year: 2024, Event: Canadian Grand Prix
Current Year: 2024, Event: Spanish Grand Prix
Current Year: 2024, Event: Austrian Grand Prix
Current Year: 2024, Event: British Grand Prix
Current Year: 2024, Event: Hungarian Grand Prix
Current Year: 2024, Event: Belgian Grand Prix
Current Year: 2024, Event: Dutch Grand Prix
Current Year: 2024, Event: Italian Grand Prix
Current Year: 2024, Event: Azerbaijan Grand Prix


Processing Years: 100%|██████████| 3/3 [00:00<00:00,  4.22it/s]

Current Year: 2024, Event: Singapore Grand Prix
Current Year: 2024, Event: United States Grand Prix
Current Year: 2024, Event: Mexico City Grand Prix
Current Year: 2024, Event: São Paulo Grand Prix
Current Year: 2024, Event: Las Vegas Grand Prix
Current Year: 2024, Event: Qatar Grand Prix
Current Year: 2024, Event: Abu Dhabi Grand Prix
Successfully created training data with 49750 samples.
  Driver  LapNumber  LapTime_last_year  LapTime_current_year
0    HAM        1.0            119.538               101.555
1    HAM        2.0            142.712                99.002
2    HAM        4.0            104.932                98.892
3    HAM        5.0            105.139                98.923
4    HAM        6.0             96.169                99.707


In [13]:
training_data

,Driver,LapNumber,LapTime_last_year,LapTime_current_year
0,HAM,1.0,119.538,101.555
1,HAM,2.0,142.712,99.002
2,HAM,4.0,104.932,98.892
3,HAM,5.0,105.139,98.923
4,HAM,6.0,96.169,99.707
...,...,...,...,...
49745,MAG,53.0,90.445,90.215
49746,MAG,54.0,90.207,91.204
49747,MAG,55.0,90.252,91.958
49748,MAG,56.0,90.429,112.994


In [77]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

def data_label_split(df_windows: pd.DataFrame):
	X = df_windows.iloc[:, :-1]
	y = df_windows.iloc[:, -1]
	return X, y

def normalize_features(X: pd.DataFrame):
   scaler = StandardScaler()
   X_normalized = scaler.fit_transform(X)
   return X_normalized


In [78]:
data = training_data[["LapTime_last_year", "LapTime_current_year"]]
X, y = data_label_split(data)
X = normalize_features(X)
X.shape, y.shape

((49750, 1), (49750,))

In [121]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
X_train.shape

(39800, 1)

In [122]:
params = {'colsample_bytree': 1.0, 
          'learning_rate': 0.05,
          'max_depth': 8, 
          'n_estimators': 250, 
          'subsample': 1.0}
model = xgb.XGBRegressor(**params)
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,1.0
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [123]:
y_pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MAE: 6.900046664296827
R² Score: 0.04772338180619051


In [124]:
pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

,Actual,Predicted
38323,93.192,94.523338
34617,99.889,82.908905
38469,89.925,92.938004
42934,84.640,87.725098
5431,89.144,72.281792
...,...,...
20457,73.517,74.364510
37692,71.821,71.449776
17538,80.443,95.181763
12104,77.508,99.192268
